In [2]:
import numpy as np
import pandas as pd
from cmdstanpy import write_stan_json
#
from neural_priors.utils.data import get_all_behavioral_data

In [27]:
def make_stan_dict(data):
    dat = data[np.isfinite(data.response)]
    subjects = dat.index.get_level_values('subject').unique()
    S = len(subjects)
    N = len(dat)
    c = dat.range.apply(lambda x: {'narrow':1, 'wide':2}[x]).values
    x = dat['n'].values.astype(int)
    y = dat['response'].values
    subject = ( dat.index.get_level_values('subject').astype('category').codes + 1).astype('int')  # Ensure subject IDs are in [1, S] # !! this means that Stan subject ids will be different

    # Prepare data dictionary for Stan
    stan_data = {
        'S': S,
        'N': N,
        'c': c,
        'x': x,
        'y': y,
        'subject': subject
    }

    return stan_data

In [3]:
bids_folder = '../../ds-neuralpriors'
beh = get_all_behavioral_data(bids_folder=bids_folder)

In [26]:
beh[np.isfinite(beh.response)] #.index.get_level_values('subject')

onset  phase  response  nr_frames     n  \
subject session run trial_nr                                                 
01      1       1   2          76.016292      5      19.0       31.0  23.0   
                    3          87.026471      5      25.0       30.0  23.0   
                    4          97.085656      5      24.0       30.0  22.0   
                    5         106.744768      5      17.0       31.0  21.0   
                    6         115.435967      5      15.0       30.0  20.0   
...                                  ...    ...       ...        ...   ...   
41      2       8   26        275.797056      5      25.0       30.0  24.0   
                    27        286.857371      5      24.0       30.0  19.0   
                    28        296.683059      5      22.0       30.0  17.0   
                    29        307.476458      5      19.0       30.0  19.0   
                    30        317.202043      5      19.0       30.0  19.0   

                              jitter  start_marker_position  response_time  \
subject session run trial_nr                                                 
01      1       1   2            4.0                   14.0       1.317963   
                    3            6.0                   11.0       0.834244   
                    4            5.0                   22.0       1.701381   
                    5            5.0                   22.0       2.018881   
                    6            4.0                   21.0       1.668272   
...                              ...                    ...            ...   
41      2       8   26           4.0                   12.0       1.635028   
                    27           6.0                   15.0       1.701679   
                    28           5.0                   15.0       0.100000   
                    29           6.0                   20.0       1.751607   
                    30           5.0                   24.0       0.049945   

                               onset_abs  duration   range  error  abs_error  \
subject session run trial_nr                                                   
01      1       1   2          83.568595  0.517188  narrow   -4.0        4.0   
                    3          94.578775  0.500469  narrow    2.0        2.0   
                    4         104.637959  0.500559  narrow    2.0        2.0   
                    5         114.297072  0.517009  narrow   -4.0        4.0   
                    6         122.988270  0.500534  narrow   -5.0        5.0   
...                                  ...       ...     ...    ...        ...   
41      2       8   26        278.746204  0.500532  narrow    1.0        1.0   
                    27        289.806518  0.500510  narrow    5.0        5.0   
                    28        299.632207  0.500525  narrow    5.0        5.0   
                    29        310.425606  0.500458  narrow    0.0        0.0   
                    30        320.151190  0.500598  narrow    0.0        0.0   

                              squared_error  
subject session run trial_nr                 
01      1       1   2                  16.0  
                    3                   4.0  
                    4                   4.0  
                    5                  16.0  
                    6                  25.0  
...                                     ...  
41      2       8   26                  1.0  
                    27                 25.0  
                    28                 25.0  
                    29                  0.0  
                    30                  0.0  

[18350 rows x 14 columns]

In [28]:
stan_dict = make_stan_dict(beh)
write_stan_json(path='data_for_stan.json', data=stan_dict)

In [22]:
def make_inits_dict(S):
    init_values = {
            'm_0_15': np.arange(10., 25.1),
            'm_0_30': np.arange(10., 40.1),
            'tau': 2.,
            'z_m_s_15': np.full((S, 16), 0.),
            'z_m_s_30': np.full((S, 31), 0.),
            #'m_s_20': np.tile( np.arange(50., 70.1), reps=(S,1) ),
            #'m_s_40': np.tile( np.arange(40., 80.1), reps=(S,1) ),
            'sigma_0_15': np.ones(16)*2.,
            'sigma_0_30': np.ones(31)*3.5,
            'nu': 1.,
            #'sigma_s_20': np.full((S, 21), 5.),
            #'sigma_s_40': np.full((S, 41), 7.),
        }
    return init_values
    


In [24]:
len( beh.index.get_level_values('subject').unique() )

39

In [25]:
init_values = make_inits_dict(39)
write_stan_json(path='inits_stan.json', data=init_values)